# Code Regridding
This notebook handles the re-gridding process necessary on the IVT files for use with the three Atmospheric River Detection Tools (ARDTs) used.

In [ ]:
import xarray as xa
from glob import glob
import xesmf as xe
import numpy as np
import pandas as pd
import cftime
import dask
from dask.distributed import Client, LocalCluster
import netCDF4
import gc

In [ ]:
# This process is sped up via a local dask cluster. Make sure you request enough memory.
cluster = LocalCluster(n_workers=16, memory_limit="15GB")
client = Client(cluster)
client

In [ ]:
# This cell grabs the lat/lon metadata needed as its not in the original IVT files.
wusd3 = pd.read_csv('../wusd3_gcms.csv', index_col=0)
meta = xa.open_dataset('/glade/campaign/uwyo/wyom0169/wus-d3/postprocess/meta/meta_new/wrfinput_d01')
ivtpath = '/glade/campaign/uwyo/wyom0169/wus-d3/postprocess/'
landmask = meta['LANDMASK'][0] # Necessary for the landfall detection piece.

In [ ]:
# Function for processing the regridding. Have to add specific dimensions for compatibility with one specific ARDT.
def regrid_ivt_data(ivtfile):
    wlon = -140
    elon = -110
    nlat = 50
    slat = 30
    degree_spacing_x = 0.1
    degree_spacing_y = 0.1
    
    x = np.arange(wlon, elon+degree_spacing_x, degree_spacing_x)
    y = np.arange(slat, nlat+degree_spacing_y, degree_spacing_y)
    xx, yy = np.meshgrid(x, y)

    # Bare-bones dataset with the re-gridded lats and lons.
    #     Important for creating the re-gridder object
    ds_out = xa.Dataset(
        {
            "lat": (["lat"], y, {"units": "degrees_north"}),
            "lon": (["lon"], x, {"units": "degrees_east"}),
            "lev": (["lev"], [1]),
            "ens": (["ens"], [1])
        }
    )

    # Create the re-gridded objects. Bilinear for IVT and landmask.
    #     This will turn landmask from 0 to 1 but the algorithm only
    #     requries > 0.5 landfrac so it's okay.
    regrid_bilinear = xe.Regridder(ivtfile, ds_out, "bilinear")
    
    dr_out_ivt = regrid_bilinear(ivtfile, keep_attrs=True)
    return dr_out_ivt

In [ ]:
# Functionally similar to regrid_ivt_data above but we don't need the extra dimensions.
def regrid_landmask(landfile):
    wlon = -140
    elon = -110
    nlat = 50
    slat = 30
    degree_spacing_x = 0.1
    degree_spacing_y = 0.1
    
    x = np.arange(wlon, elon+degree_spacing_x, degree_spacing_x)
    y = np.arange(slat, nlat+degree_spacing_y, degree_spacing_y)
    xx, yy = np.meshgrid(x, y)

    ds_out = xa.Dataset(
        {
            "lat": (["lat"], y, {"units": "degrees_north"}),
            "lon": (["lon"], x, {"units": "degrees_east"}),
        }
    )

    regrid_bilinear = xe.Regridder(landfile, ds_out, "bilinear")
    
    dr_out_land = regrid_bilinear(landfile, keep_attrs=True)
    return dr_out_land

In [ ]:
# This function standardizes the data processing of IVT files.
def ivt_file(ivt):
    ivtx = ivt.sel(component=0)
    ivty = ivt.sel(component=1)
    lat = meta.XLAT.squeeze().rename({'south_north':'lat2d', 'west_east':'lon2d'}) # Rename lat/lon dims to match the WUS-D3 names
    lon = meta.XLONG.squeeze().rename({'south_north':'lat2d', 'west_east':'lon2d'})
    time = ivt.day # Original IVT files have 'day' as the time dimension but renaming to 'time' just simplifies a lot downstream.
    
    ivt_xy = ivtx.expand_dims(["lev", "ens"]).transpose('lon2d', 'lat2d', 'lev', 'day', 'ens').rename({'ivt':'ivtx'}) # Adds the necessary dimensions for one of the ARDTs.
    ivt_y = ivty.expand_dims(["lev", "ens"]).transpose('lon2d', 'lat2d', 'lev', 'day', 'ens').rename({'ivt':'ivty'})  # Above, but for the y direction
    ivt_xy['ivty'] = ivt_y['ivty'] # Effectively joins the ivtx and ivty files together.
    ivt_xy = ivt_xy.assign_coords({'lon': lon, 'lat': lat, 'lev': [1], 'ens':[1]}).drop(['XLAT','XLONG'])
    ivt_xy_reg = regrid_ivt_data(ivt_xy) # Regridding function
    ivt_xy_reg = ivt_xy_reg.transpose('ens', 'day', 'lev', 'lat', 'lon') # MATLAB order dimensionss reverse of python, necessary for tARget
    ivt_xy_reg = ivt_xy_reg.rename({'day': 'time'}) # Original IVT files have 'day' as the time dimension but renaming to 'time' just simplifies a lot downstream.
    
    return ivt_xy_reg

In [ ]:
%%time

# regridding_models function
def regridding_models(model, member):
    if model != 'era5':
        # Join the historical and future files together to avoid downstream confustion.
        extpath_historical = glob(ivtpath+f"{model}_{member}_historical/postprocess/d01/ivt.daily.{model}.{member}.hist.d01.*.nc")
        extpath_ssp370 = glob(ivtpath+f"{model}_{member}_ssp370/postprocess/d01/ivt.daily.{model}.{member}.ssp370.d01.*.nc")
        extpath = [*extpath_historical, *extpath_ssp370]
    else:
        extpath = glob(ivtpath+'era5/postprocess/d01/ivt.daily.era5.d01.*.nc')

        
    print(f"{model}/{member} begin.")
    ivt = xa.open_mfdataset(extpath, chunks={'day': 365}, parallel=True) # Chunk data to run in parallel
    regrid_ivt = ivt_file(ivt) # The regridding/processing function is nested inside here.
    if model == 'era5':
        regrid_ivt.to_netcdf(f'/glade/derecho/scratch/tcorrie/regrids/ivt.daily.era5.nan.d01_regridded.nc')
    else:
        regrid_ivt.to_netcdf(f'/glade/derecho/scratch/tcorrie/regrids/ivt.daily.{model}.{member}.d01_regridded.nc')
    print(f"{model}/{member} finished.")
    gc.collect() # Clear unused memory for the dask workers

# Similar to the above function but for the CESM2-LE, the filepath structure is a bit different.
def regridding_ensemble(idx):
    member=str(idx)
    extpath_ancient = glob(ivtpath+f"cesm2-le/1850_1979/{member}/d01/ivt.daily.cesm2-le.{member}.bias-correct.d01.*.nc")
    extpath_modern = glob(ivtpath+f"cesm2-le/{member}/d01/ivt.daily.cesm2-le.{member}.bias-correct.d01.*.nc")
    allfiles = [*extpath_ancient, *extpath_modern]
    print(f"cesm2-le/{member} begin.")
    ivt = xa.open_mfdataset(allfiles, chunks={'day': 365}, parallel=True)
    regrid_ivt = ivt_file(ivt)
    regrid_ivt.to_netcdf(f'/glade/derecho/scratch/tcorrie/regrids/ivt.daily.cesm2-le.{member}.d01_regridded.nc')
    print(f"cesm2-le/{member} finished.")
    gc.collect()

# Loops in serial but the regridding per model is parallelized (hopefully)
for i in range(len(wusd3.Model)):
    model = wusd3.iat[i, 0]
    member = wusd3.iat[i, 2]
    cal = wusd3.iat[i, 3]
    regridding_models(model, member)

for i in range(1011, 1201, 20):
    regridding_ensemble(i)

In [ ]:
# Land regridding is its own file.
land_regrd = regrid_landmask(landmask)
land_regrd.to_netcdf('/glade/derecho/scratch/tcorrie/regrids/landmask_regridded.nc')

In [ ]:
client.shutdown() # Might not be absolutely necessary to shut down a client everytime but it's here if you need it.